# Compression: trade-offs between speed and size

In this demo and exercise, we explore the trade-off between reading and writing speed, and the size of the data on disk. Again, we use the two-photon data we wrote to a Zarr file on Monday.

We start by lazily opening the two-photon data (in read-only mode, `mode="r"`) with `zarr-python`, which allows us to inspect the compression information. 

In [ ]:
from pathlib import Path

import zarr

in_path = Path("./data/twophoton_series.zarr")

two_photon_series_zarr = zarr.open(in_path, mode="r")
print(two_photon_series_zarr.info)

Because we wrote this earlier in the course, we just used the default zarr compression: a Zstd Codec with compression level 0. We now want to write this same array back to disk, but with the Blosc codec and varying compression levels. We start by reading the same array into `dask`, because this will allow us to easily copy it chunk-by-chunk

In [ ]:
import dask.array as da

two_photon_series = da.from_zarr(
    in_path, mode="r", chunks=two_photon_series_zarr.chunks
)
two_photon_series

Next, we loop over different compessions levels (0,3,6), and for each compression level, we

1. open a Zarr file on disk, specifying the compressor we want
2. write the array to disk chunk by chunk
3. record how much time it took
4. record how big the Zarr file is on disk

In [ ]:
import time

from zarr.codecs import BloscCodec

from course_large_array_data import zarr_disk_size

write_times = []
sizes_on_disk = []
compression_levels = range(0, 9, 3)

for clevel in compression_levels:
    start = time.time()

    # step 1.
    outpath = Path(f"./data/two_photon_series_clevel-{clevel}.zarr")
    destination_on_disk = zarr.create_array(
        outpath,
        shape=two_photon_series.shape,
        dtype=two_photon_series.dtype,
        chunks=two_photon_series.chunksize,
        zarr_format=3,
        overwrite=True,
        compressors=[
            BloscCodec(cname="zstd", clevel=clevel),
        ],
    )

    # step 2.
    two_photon_series.to_zarr(
        destination_on_disk,
        compute=True,
        mode="w",
    )

    # step 3.
    stop = time.time()
    print(f"Elapsed: {stop - start}")
    write_times.append(stop - start)

    # step 4.
    size_on_disk = zarr_disk_size(outpath) / 1024**2
    print(f"{size_on_disk:.2f} MiB")
    sizes_on_disk.append(size_on_disk)

We can now use our results to plot the trade-off between size-on-disk and writing speed.

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.subplot(1, 2, 1)
plt.plot(compression_levels, write_times)
plt.title("Write times (s)")
plt.subplot(1, 2, 2)
plt.plot(compression_levels, sizes_on_disk)
plt.title("Size on disk (MB)")
plt.show()

## Stretch exercise

Modify the code above to also compute and plot read times with varying compression levels, or vary the [zarr codec used.](https://zarr.readthedocs.io/en/stable/api/zarr/codecs/)

Alternatively, feel free to try out some of the concepts presented here on your own data, invent your own related stretch exercise or help others in the room.

## Key take-aways

* We can compress data using different codecs, and varying the compression levels
* The right choice will depend on the data itself
* Higher compression levels come with small size on disk (more compressed) but also slower reading/writing speeds.